# Módulo 20 · Práctica: del modelo base al asistente con LoRA

Repites en miniatura lo que se hace con un LLM tras el preentrenamiento. Un mini-GPT de caracteres que solo sabe **continuar texto** aprende a **responder instrucciones** `p: … / r: …` con pérdida enmascarada y **LoRA programada a mano**. Corre en CPU en menos de un minuto.

| Parte | Qué haces |
| --- | --- |
| 1 | Preentrenar un mini-GPT de caracteres |
| 2 | Dataset de instrucciones y enmascarado de la pérdida |
| 3 | LoRA a mano: capa, conteo de parámetros, inicialización B = 0 |
| 4 | SFT con LoRA frente a ajuste completo |
| 5 | Olvido catastrófico, adaptadores desmontables y *replay* |
| 6 | Fusionar el adaptador y mirar ΔW con la SVD |
| 7 | Retos |

Las celdas `# TU CÓDIGO` son ejercicios; la celda `# Comprobación` siguiente te dice si están bien.

## Parte 1 · Un mini-GPT preentrenado

El corpus es un texto corto en español escrito para el curso. El vocabulario son caracteres más un token especial `<fin>` (índice 0), que hará de fin de turno.

In [ ]:
import math, random, time, copy
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); random.seed(0)
t_inicio = time.time()

texto = """el sol sale por el este y se pone por el oeste. la luna tarda casi un mes en dar una vuelta a la tierra.
en la cocina hay pan, queso, aceite y tomate. mi abuela prepara una tortilla de patatas cada domingo.
el tren sale a las ocho y llega a las diez. los viajeros leen, duermen o miran por la ventana.
un metro tiene cien centimetros y un kilo tiene mil gramos. una hora tiene sesenta minutos.
si tengo 2 manzanas y compro 3 mas, tengo 5. si tengo 4 euros y gano 4, tengo 8 euros.
el gato duerme en el sofa mientras el perro juega en el jardin con una pelota roja.
madrid es la capital de espana, paris es la capital de francia y roma es la capital de italia.
en invierno hace frio y en verano hace calor. de dia hay luz y de noche hay estrellas.
la biblioteca abre a las 9 y cierra a las 21. alli estudian alumnos de fisica y de historia.
en el mercado se cuenta 1, 2, 3, 4, 5, 6, 7, 8, 9 y 10, y luego 11, 12, 13, 14, 15, 16, 17 y 18.
hola, buenos dias. gracias por venir. hasta luego y buenas noches.
"""
vocab = ["<fin>"] + list("\n abcdefghijklmnopqrstuvwxyzñ0123456789.,:?¿+=")
stoi = {c: i for i, c in enumerate(vocab)}
FIN, V, CTX = 0, len(vocab), 40
def encode(s): return [stoi[c] for c in s]
def decode(ids): return "".join(vocab[i] for i in ids if i != FIN)
data = torch.tensor(encode(texto))
print("vocabulario:", V, "· caracteres del corpus:", len(data))

In [ ]:
class Bloque(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h = h
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.q, self.k, self.v, self.o = (nn.Linear(d, d, bias=False) for _ in range(4))
        self.fc1, self.fc2 = nn.Linear(d, 4 * d, bias=False), nn.Linear(4 * d, d, bias=False)
    def forward(self, x):
        B, T, D = x.shape
        y = self.ln1(x)
        sep = lambda t: t.view(B, T, self.h, D // self.h).transpose(1, 2)
        a = F.scaled_dot_product_attention(sep(self.q(y)), sep(self.k(y)), sep(self.v(y)), is_causal=True)
        x = x + self.o(a.transpose(1, 2).reshape(B, T, D))
        return x + self.fc2(F.gelu(self.fc1(self.ln2(x))))

class MiniGPT(nn.Module):
    def __init__(self, V, d=64, h=4, L=2):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(CTX, d)
        self.bloques = nn.ModuleList([Bloque(d, h) for _ in range(L)])
        self.lnf, self.cabeza = nn.LayerNorm(d), nn.Linear(d, V, bias=False)
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1]))
        for b in self.bloques:
            x = b(x)
        return self.cabeza(self.lnf(x))

def n_entrenables(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

base = MiniGPT(V)
print("parámetros del modelo base:", n_entrenables(base))

In [ ]:
def lote_texto(bs=16):
    i = torch.randint(0, len(data) - CTX - 1, (bs,))
    x = torch.stack([data[j:j + CTX] for j in i])
    y = torch.stack([data[j + 1:j + CTX + 1] for j in i])
    return x, y

opt = torch.optim.AdamW(base.parameters(), lr=3e-3)
for paso in range(500):
    x, y = lote_texto()
    loss = F.cross_entropy(base(x).reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if paso % 100 == 0:
        print(f"paso {paso:3d} · pérdida {loss.item():.3f}")

@torch.no_grad()
def perdida_texto(m):
    g = torch.random.get_rng_state(); torch.manual_seed(123)
    tot = sum(F.cross_entropy(m(x).reshape(-1, V), y.reshape(-1)).item()
              for x, y in (lote_texto(64) for _ in range(10)))
    torch.random.set_rng_state(g)
    return tot / 10

L_base = perdida_texto(base)
print("pérdida de texto del modelo base:", round(L_base, 3))

Pregúntale algo al modelo base en el formato de chat. Verás que **no responde**: continúa el texto como en el corpus y nunca emite `<fin>`. Ese es el punto de partida de cualquier LLM antes del SFT.

In [ ]:
@torch.no_grad()
def responder(m, pregunta, maxn=24):
    # greedy; devuelve (respuesta, ¿terminó con <fin>?)
    ids = encode(f"p: {pregunta}\nr:")
    for _ in range(maxn):
        j = int(m(torch.tensor([ids[-CTX:]]))[0, -1].argmax())
        if j == FIN:
            return decode(ids).split("\nr:", 1)[1].strip(), True
        ids.append(j)
    return decode(ids).split("\nr:", 1)[1].strip(), False

for q in ["capital de francia", "doble de 3", "primera letra de gato"]:
    print(q, "->", responder(base, q))

## Parte 2 · Instrucciones y enmascarado

El dataset es sintético: letras de palabras, números escritos, dobles y algunas respuestas fijas. Para medir **generalización**, separamos 20 palabras: sus preguntas («primera letra de …», «última letra de …») no se ven en el entrenamiento.

Cada ejemplo se convierte en la secuencia `p: pregunta\nr: respuesta<fin>`. La pérdida solo cuenta los tokens de la respuesta (incluido `<fin>`). En PyTorch se consigue poniendo la etiqueta `-100` en el resto: `F.cross_entropy` la ignora.

In [ ]:
nums = "cero uno dos tres cuatro cinco seis siete ocho nueve".split()
palabras = sorted({w.strip(".,") for w in texto.split()
                   if w.strip(".,").isalpha() and 3 <= len(w.strip(".,")) <= 8})
ejemplos = []
for w in palabras:
    ejemplos += [(f"primera letra de {w}", w[0]), (f"ultima letra de {w}", w[-1])]
for n in range(10):
    ejemplos += [(f"escribe {n} con letras", nums[n]), (f"doble de {n}", str(2 * n))]
ejemplos += [("capital de espana", "madrid"), ("capital de francia", "paris"),
             ("capital de italia", "roma"), ("saluda", "hola, ¿en que te ayudo?"),
             ("contrario de frio", "calor"), ("contrario de dia", "noche"), ("despidete", "hasta luego")]
rng = random.Random(0); rng.shuffle(palabras)
ocultas = set(palabras[:20])
test = [e for e in ejemplos if "letra" in e[0] and e[0].split()[-1] in ocultas]
train = [e for e in ejemplos if e not in test]
print(len(train), "de entrenamiento ·", len(test), "de prueba")
print(train[:4])

**Ejercicio 1.** Completa `construir(p, r)`. Debe devolver `(entrada, etiquetas)` desplazadas un token, como en el preentrenamiento, pero con `-100` en todas las posiciones cuyo objetivo pertenezca al prompt `p: …\nr:`.

Pista: forma `ids = prompt + respuesta + [FIN]` y `etiq = [-100] * len(prompt) + respuesta + [FIN]`; devuelve `ids[:-1]` y `etiq[1:]`.

In [ ]:
def construir(p, r):
    prompt = encode(f"p: {p}\nr:")
    resp = encode(" " + r)
    # TU CÓDIGO
    return None, None

In [ ]:
# Comprobación
x_, y_ = construir("doble de 2", "4")
assert x_ is not None, "construir devuelve None"
assert len(x_) == len(y_) == len("p: doble de 2\nr: 4"), "Longitud incorrecta: entrada y etiquetas deben medir len(texto)"
assert y_[-3:] == [stoi[" "], stoi["4"], FIN], "Las tres últimas etiquetas deben ser ' ', '4' y <fin>"
assert all(t == -100 for t in y_[:-3]), "Todas las etiquetas del prompt deben valer -100"
assert x_[0] == stoi["p"] and x_[-1] == stoi["4"], "La entrada debe ir de 'p' al último carácter de la respuesta"
print("Correcto: de", len(y_), "posiciones, solo", sum(t != -100 for t in y_), "cuentan en la pérdida")

In [ ]:
def lote_sft(lista, bs=32):
    pares = [construir(p, r) for p, r in random.choices(lista, k=bs)]
    T = max(len(i) for i, _ in pares)
    X = torch.full((bs, T), FIN)          # relleno con <fin> ...
    Y = torch.full((bs, T), -100)         # ... que tampoco cuenta en la pérdida
    for k, (i, e) in enumerate(pares):
        X[k, :len(i)] = torch.tensor(i); Y[k, :len(e)] = torch.tensor(e)
    return X, Y

X, Y = lote_sft(train, 4)
print(X.shape, "· fracción de posiciones que cuentan:", (Y != -100).float().mean().item())

## Parte 3 · LoRA a mano

LoRA congela W y aprende una corrección de bajo rango: y = W x + (α/r) · B A x, con A de r × k y B de d × r. A empieza aleatoria y **B empieza a cero**, así que al principio el modelo ajustado es idéntico al base.

$$W' = W + \frac{\alpha}{r} B A, \qquad \#\text{entrenables} = r(d + k)$$

**Ejercicio 2.** Escribe `params_lora(d, k, r)` y calcula cuántos parámetros entrenables añade LoRA con r = 4 a las seis matrices (q, k, v, o, fc1, fc2) de los dos bloques del mini-GPT (d = 64, MLP de 256).

In [ ]:
def params_lora(d, k, r):
    # TU CÓDIGO
    return None

total_lora_r4 = None  # TU CÓDIGO: suma sobre las 6 matrices y los 2 bloques
print(total_lora_r4)

In [ ]:
# Comprobación
assert params_lora(4096, 4096, 8) == 65_536, "params_lora(4096, 4096, 8) debe valer 65.536"
assert total_lora_r4 == 9_216, f"Se esperaban 9.216 parámetros LoRA y obtienes {total_lora_r4}"
print("Correcto:", total_lora_r4, "frente a", n_entrenables(base), f"({100 * total_lora_r4 / n_entrenables(base):.2f} %)")

**Ejercicio 3.** Completa `forward` de `LoRALinear`. Calcula primero `x @ A.T` (tamaño r) y luego `@ B.T`: así nunca formas la matriz d × k completa. Si `self.activo` es `False`, devuelve solo la salida de la capa base.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_lin, r=4, alpha=8):
        super().__init__()
        self.base = base_lin
        self.escala = alpha / r
        d, k = base_lin.out_features, base_lin.in_features
        self.A = nn.Parameter(torch.randn(r, k) / math.sqrt(k))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.activo = True
    def forward(self, x):
        # TU CÓDIGO
        return self.base(x)

def aplicar_lora(modelo, r=4, alpha=8):
    m = copy.deepcopy(modelo)
    for p in m.parameters():
        p.requires_grad = False
    for b in m.bloques:
        for nombre in ["q", "k", "v", "o", "fc1", "fc2"]:
            setattr(b, nombre, LoRALinear(getattr(b, nombre), r, alpha))
    return m

def activar_lora(m, valor):
    for mod in m.modules():
        if isinstance(mod, LoRALinear):
            mod.activo = valor

In [ ]:
# Comprobación
torch.manual_seed(1)
capa = LoRALinear(nn.Linear(5, 3, bias=False), r=2, alpha=4)
xx = torch.randn(7, 5)
assert torch.allclose(capa(xx), capa.base(xx)), "Con B = 0, la salida debe coincidir con la capa base"
with torch.no_grad():
    capa.B.normal_()
esperado = capa.base(xx) + 2.0 * xx @ (capa.B @ capa.A).T
assert torch.allclose(capa(xx), esperado, atol=1e-5), "La salida debe ser W x + (alpha/r) B A x"
capa.activo = False
assert torch.allclose(capa(xx), capa.base(xx)), "Con activo=False debe devolver solo la capa base"
m_prueba = aplicar_lora(base, r=4)
assert n_entrenables(m_prueba) == 9_216, "aplicar_lora con r=4 debe dejar 9.216 parámetros entrenables"
X, _ = lote_sft(train, 4)
assert torch.allclose(m_prueba(X), base(X), atol=1e-5), "Recién creado, el modelo LoRA debe ser idéntico al base"
print("Correcto: LoRA arranca exactamente en el modelo base")

## Parte 4 · SFT: LoRA frente a ajuste completo

Entrenamos dos copias del modelo base con los mismos datos y pasos: una con LoRA (r = 4, α = 8) y otra actualizando los 107.520 parámetros. Medimos exactitud (respuesta exacta), si terminan con `<fin>` y la pérdida sobre el texto original.

In [ ]:
def sft(m, pasos=600, lr=1e-2, replay=0.0):
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=lr, weight_decay=0.0)
    for paso in range(pasos):
        X, Y = lote_sft(train)
        loss = F.cross_entropy(m(X).reshape(-1, V), Y.reshape(-1), ignore_index=-100)
        if replay > 0:
            x, y = lote_texto()
            loss = loss + replay * F.cross_entropy(m(x).reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()

def evaluar(m, lista):
    res = [responder(m, p) for p, _ in lista]
    exact = sum(g == r for (g, _), (_, r) in zip(res, lista)) / len(lista)
    fin = sum(f for _, f in res) / len(lista)
    return round(exact, 3), round(fin, 3)

tabla = {"base": (n_entrenables(base), evaluar(base, train), evaluar(base, test), L_base)}

torch.manual_seed(1); random.seed(1)
m_lora = aplicar_lora(base, r=4, alpha=8)
t = time.time(); sft(m_lora, lr=1e-2); t_lora = time.time() - t
tabla["LoRA r=4"] = (n_entrenables(m_lora), evaluar(m_lora, train), evaluar(m_lora, test), perdida_texto(m_lora))

torch.manual_seed(1); random.seed(1)
m_full = copy.deepcopy(base)
t = time.time(); sft(m_full, lr=1e-3); t_full = time.time() - t
tabla["completo"] = (n_entrenables(m_full), evaluar(m_full, train), evaluar(m_full, test), perdida_texto(m_full))

print(f"{'modelo':10s} {'entrenables':>11s}  (exact, fin) train  (exact, fin) test  pérdida texto")
for k, (n, tr, te, lt) in tabla.items():
    print(f"{k:10s} {n:11d}  {str(tr):18s} {str(te):17s} {lt:.3f}")
print(f"tiempo SFT: LoRA {t_lora:.1f} s · completo {t_full:.1f} s")

In [ ]:
for q, r in test[:8]:
    print(f"{q:28s} esperado {r!r:5s} LoRA {responder(m_lora, q)[0]!r:8s} completo {responder(m_full, q)[0]!r}")

**Qué observar.** Ambos aprenden el **formato** (terminan con `<fin>`). En las palabras ocultas aciertan solo una parte: con 1.000 caracteres de preentrenamiento hay poco que reaprovechar.

## Parte 5 · Olvido catastrófico

La pérdida sobre el texto original ha subido mucho: el modelo ajustado continúa peor el corpus que antes sabía. Con LoRA, W no ha cambiado: **apagar el adaptador** recupera exactamente el modelo base.

**Ejercicio 4.** Apaga el adaptador de `m_lora` con `activar_lora`, guarda su pérdida de texto en `L_apagado` y vuelve a encenderlo.

In [ ]:
# TU CÓDIGO
L_apagado = None

In [ ]:
# Comprobación
assert L_apagado is not None, "Calcula L_apagado"
assert abs(L_apagado - L_base) < 1e-4, "Con el adaptador apagado, la pérdida debe ser la del modelo base"
assert all(mod.activo for mod in m_lora.modules() if isinstance(mod, LoRALinear)), "Vuelve a encender el adaptador"
print(f"Correcto: encendido {tabla['LoRA r=4'][3]:.3f} · apagado {L_apagado:.3f} · base {L_base:.3f}")

Otra defensa es el *replay*: mezclar en cada paso algo de pérdida de preentrenamiento. Entrena una tercera copia LoRA con `replay=1.0`.

In [ ]:
torch.manual_seed(1); random.seed(1)
m_replay = aplicar_lora(base, r=4, alpha=8)
sft(m_replay, lr=1e-2, replay=1.0)
L_replay = perdida_texto(m_replay)
print("LoRA + replay · train", evaluar(m_replay, train), "· test", evaluar(m_replay, test), f"· pérdida texto {L_replay:.3f}")

In [ ]:
# Comprobación
assert L_replay < tabla["LoRA r=4"][3] / 2, "El replay debería reducir mucho el olvido"
assert evaluar(m_replay, train)[0] > 0.8, "Y mantener la exactitud de entrenamiento"
print("Correcto: el replay conserva la instrucción y olvida mucho menos")

## Parte 6 · Fusionar el adaptador y mirar ΔW

Tras entrenar, puedes **fusionar**: W ← W + (α/r) B A. El modelo resultante es un MiniGPT normal, sin coste extra en inferencia.

**Ejercicio 5.** Completa `fusionar`: crea una copia de `base` y suma a cada matriz `nn.Linear` el término `escala * B @ A` del `LoRALinear` correspondiente de `m_lora`.

In [ ]:
@torch.no_grad()
def fusionar(m_lora, base):
    m = copy.deepcopy(base)
    for b_m, b_l in zip(m.bloques, m_lora.bloques):
        for nombre in ["q", "k", "v", "o", "fc1", "fc2"]:
            capa = getattr(b_l, nombre)
            # TU CÓDIGO: suma la corrección a getattr(b_m, nombre).weight
    return m

m_fus = fusionar(m_lora, base)

In [ ]:
# Comprobación
X, _ = lote_sft(test, 8)
assert not torch.allclose(m_fus(X), base(X), atol=1e-3), "El modelo fusionado no puede ser igual al base"
assert torch.allclose(m_fus(X), m_lora(X), atol=1e-4), "El modelo fusionado debe dar las mismas salidas que el LoRA"
print("Correcto: fusionado y LoRA coinciden; parámetros del fusionado:", sum(p.numel() for p in m_fus.parameters()))

Conexión con el módulo 3: el rango de ΔW = (α/r)·B·A es como mucho r. Comparamos con el ΔW del ajuste completo, que puede tener rango 64. ¿Qué fracción de su «energía» (Σσᵢ²) capturan sus 4 primeros valores singulares?

In [ ]:
with torch.no_grad():
    dW_lora = m_fus.bloques[0].q.weight - base.bloques[0].q.weight
    dW_full = m_full.bloques[0].q.weight - base.bloques[0].q.weight
print("rango ΔW LoRA:", torch.linalg.matrix_rank(dW_lora).item(), "· rango ΔW completo:", torch.linalg.matrix_rank(dW_full).item())
s = torch.linalg.svdvals(dW_full)
energia4 = (s[:4] ** 2).sum() / (s ** 2).sum()
print(f"energía de ΔW completo en los 4 primeros valores singulares: {energia4:.3f}")
print("σ₁…σ₈ =", [round(v, 3) for v in s[:8].tolist()])

## Parte 7 · Retos

1. **Rango.** Repite la Parte 4 con r = 1, 2, 8, 16 (α = 2r) y dibuja exactitud de prueba frente a parámetros entrenables.
2. **Sin máscara.** Entrena con pérdida sobre toda la secuencia. ¿Qué genera el modelo si le das solo `p:`?
3. **Solo q y v**, como en el artículo de LoRA. ¿Cuántos parámetros quedan?
4. **QLoRA de juguete.** Redondea los pesos congelados a 16 niveles por fila (4 bits, módulo 19) antes de entrenar el adaptador.

In [ ]:
print(f"Tiempo total del notebook: {time.time() - t_inicio:.0f} s")

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# Ejercicio 1
#     ids = prompt + resp + [FIN]
#     etiq = [-100] * len(prompt) + resp + [FIN]
#     return ids[:-1], etiq[1:]
# Ejercicio 2
# def params_lora(d, k, r):
#     return r * (d + k)
# total_lora_r4 = 2 * (4 * params_lora(64, 64, 4) + params_lora(256, 64, 4) + params_lora(64, 256, 4))
# Ejercicio 3 (dentro de forward)
#     y = self.base(x)
#     if not self.activo:
#         return y
#     return y + self.escala * (x @ self.A.T) @ self.B.T
# Ejercicio 4
# activar_lora(m_lora, False)
# L_apagado = perdida_texto(m_lora)
# activar_lora(m_lora, True)
# Ejercicio 5 (dentro del bucle)
#             getattr(b_m, nombre).weight += capa.escala * capa.B @ capa.A